# From Logistic Regression to Feed-Forward Neural Networks
### A visual, animated walkthrough · CIS 635 Knowledge Discovery & Data Mining

This notebook builds **one tiny neural-network class from scratch in NumPy** and uses it three times, changing only the architecture:

| Part | Model | Architecture (`sizes`) | What you will see |
|---|---|---|---|
| **A** | Logistic regression | `[2, 1]` | One sigmoid neuron draws a straight line and switches **on/off** as a point crosses it |
| **B** | Two-layer network (input → output) | `[2, 3]` | Three logistic neurons side by side + softmax = multi-class, but every boundary is **still straight** |
| **C** | Three-layer network (input → **hidden** → output) | `[2, 8, 3]` | Hidden ReLU nodes **switch on and off**, each one a "mini logistic regression", and together they bend the boundary |
| **D** | Side-by-side | `[2,3]` → `[2,2,3]` → `[2,4,3]` → `[2,8,3]` | How adding hidden units changes the decision regions |

**The key message:** a logistic-regression model *is* a neural network with no hidden layer. A feed-forward network just inserts a layer of logistic-regression-like units whose outputs become new features for the final classifier.

> **How to run:** `Runtime → Run all`. Everything runs in about 2–3 minutes on a free Colab CPU. Each animation shows a ▶ player. Press play, or step through frame by frame with the arrow buttons.
>
> **Layer counting:** here we count layers of *nodes*: input, (hidden), output. So "two-layer" = input + output, and "three-layer" = input + one hidden + output. Some textbooks count only *weight* layers, which would call Part C a "two-layer network".

In [ ]:
# ============================ Setup ============================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import to_rgb
from matplotlib.patches import Circle
from IPython.display import HTML, display

plt.rcParams.update({
    "figure.dpi": 80, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "animation.embed_limit": 80,          # MB, allows long embedded animations
})

# Colors (colorblind-checked categorical palette) + marker shapes as a second cue
CLASS_COLORS  = ["#2a78d6", "#eb6834", "#1baf7a"]   # blue, orange, aqua
CLASS_MARKERS = ["o", "s", "^"]
POS_W, NEG_W  = "#4a4a4a", "#d55181"                 # positive weight: solid grey; negative: dashed magenta
ON_COLOR      = "#eda100"                             # a hidden node that is ON glows amber
OFF_FACE, OFF_EDGE = "#ececec", "#a0a0a0"

def blend(color, t):
    # Mix a color with white: t=0 -> white, t=1 -> full color
    c = np.array(to_rgb(color)); t = float(np.clip(t, 0, 1))
    return tuple((1 - t) * np.ones(3) + t * c)

def show_anim(fig, anim):
    # Render a matplotlib animation as an interactive HTML player (works in Colab)
    plt.close(fig)
    display(HTML(anim.to_jshtml(default_mode="once")))

print("Setup complete")

## 0 · Toy datasets
All data is 2-D so we can draw the input space directly.

* **Two blobs:** two classes, linearly separable (Part A)
* **Three blobs:** three classes, linearly separable (Part B)
* **Core & quadrants:** three classes that are **not** linearly separable. Class A sits in the middle, and a ring around it is cut into four arcs that alternate B, C, B, C (the stress test for Parts A–D).

In [ ]:
def make_two_blobs(n=200, seed=0):
    rng = np.random.default_rng(seed)
    X0 = rng.normal([-1.2, -0.8], 0.65, (n // 2, 2))
    X1 = rng.normal([ 1.1,  0.9], 0.65, (n // 2, 2))
    return np.vstack([X0, X1]), np.r_[np.zeros(n // 2, int), np.ones(n // 2, int)]

def make_three_blobs(n=300, seed=1):
    rng = np.random.default_rng(seed)
    centers = np.array([[-1.4, -1.0], [1.5, -0.8], [0.0, 1.5]])
    X = np.vstack([rng.normal(c, 0.55, (n // 3, 2)) for c in centers])
    return X, np.repeat(np.arange(3), n // 3)

def make_core_quadrants(n=450, seed=2):
    # class A = central core; classes B and C = four arcs of a ring, alternating B, C, B, C
    rng = np.random.default_rng(seed)
    m, gap = n // 3, 0.15
    core = rng.normal(0, 0.45, (m, 2))
    def arc(a0, k):
        t = rng.uniform(a0 + gap, a0 + np.pi / 2 - gap, k); r = rng.uniform(1.45, 2.25, k)
        return np.c_[r * np.cos(t), r * np.sin(t)]
    B = np.vstack([arc(0, m // 2), arc(np.pi, m - m // 2)])               # quadrants I and III
    C = np.vstack([arc(np.pi / 2, m // 2), arc(3 * np.pi / 2, m - m // 2)])  # quadrants II and IV
    return np.vstack([core, B, C]), np.repeat(np.arange(3), m)

X2, y2 = make_two_blobs()
X3, y3 = make_three_blobs()
Xc, yc = make_core_quadrants()
CLASS_NAMES = ["A", "B", "C"]

def plot_data(ax, X, y, alpha=0.9, s=22):
    for k in np.unique(y):
        ax.scatter(*X[y == k].T, s=s, c=CLASS_COLORS[k], marker=CLASS_MARKERS[k],
                   edgecolor="white", linewidth=0.6, alpha=alpha, label=f"class {CLASS_NAMES[k]}", zorder=3)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (X, y, t) in zip(axes, [(X2, y2, "Two blobs (Part A)"), (X3, y3, "Three blobs (Part B)"),
                                (Xc, yc, "Core & quadrants (Parts A–D)")]):
    plot_data(ax, X, y); ax.set_title(t); ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 1 · One class for every model
`Net(sizes)` is a fully connected feed-forward network:

* **hidden layers** use ReLU, $h = \max(0, z)$. A hidden node is literally **off** ($h=0$) or **on** ($h>0$).
* **output layer**: one node → **sigmoid** (binary); several nodes → **softmax** (multi-class).
* **training**: cross-entropy loss + gradients by back-propagation, optimized with Adam (gradient descent with an adaptive step size).

So `Net([2, 1])` **is** logistic regression: $\hat p = \sigma(w_1x_1 + w_2x_2 + b)$.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

class Net:
    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.W = [rng.normal(0, np.sqrt(2 / sizes[i]), (sizes[i], sizes[i + 1])) for i in range(len(sizes) - 1)]
        self.b = [np.zeros(s) for s in sizes[1:]]
        self.hidden_mask = None      # 1 = node allowed, 0 = node forced OFF (used for "knock-out" demos)
        self.hidden_scale = None     # typical max activation of each hidden node (for drawing brightness)

    @property
    def n_params(self):
        return sum(W.size for W in self.W) + sum(b.size for b in self.b)

    def copy(self):
        n = Net(self.sizes); n.W = [w.copy() for w in self.W]; n.b = [b.copy() for b in self.b]
        n.hidden_scale = self.hidden_scale; return n

    def load(self, snap):
        self.W = [w.copy() for w in snap["W"]]; self.b = [b.copy() for b in snap["b"]]; return self

    # ---------- forward pass: returns the activation of every layer ----------
    def forward(self, X, return_all=False):
        acts, a = [X], X
        for l, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            if l < len(self.W) - 1:                       # hidden layer: ReLU
                a = np.maximum(0, z)
                if self.hidden_mask is not None and l == 0:
                    a = a * self.hidden_mask
            else:                                         # output layer
                a = sigmoid(z) if z.shape[1] == 1 else softmax(z)
            acts.append(a)
        return acts if return_all else acts[-1]

    def predict_proba(self, X):
        p = self.forward(X)
        return np.c_[1 - p, p] if p.shape[1] == 1 else p

    def predict(self, X): return self.predict_proba(X).argmax(1)

    def _targets(self, y):
        return y.reshape(-1, 1).astype(float) if self.sizes[-1] == 1 else np.eye(self.sizes[-1])[y]

    def loss(self, X, y):
        P, Y = np.clip(self.forward(X), 1e-9, 1 - 1e-9), self._targets(y)
        if P.shape[1] == 1:
            return float(-np.mean(Y * np.log(P) + (1 - Y) * np.log(1 - P)))
        return float(-np.mean(np.sum(Y * np.log(P), axis=1)))

    # ---------- training with back-propagation + Adam ----------
    def fit(self, X, y, epochs=300, lr=0.05, n_frames=50):
        Y, L = self._targets(y), len(self.W)
        record = set(np.unique(np.r_[0, np.geomspace(1, epochs, n_frames - 1).astype(int)]))
        m = [np.zeros_like(p) for p in self.W + self.b]; v = [np.zeros_like(p) for p in self.W + self.b]
        history = []
        for ep in range(epochs + 1):
            acts = self.forward(X, return_all=True)
            if ep in record:
                history.append(dict(epoch=ep, W=[w.copy() for w in self.W], b=[b.copy() for b in self.b],
                                    loss=self.loss(X, y), acc=float((self.predict(X) == y).mean())))
            if ep == epochs: break
            delta = (acts[-1] - Y) / len(X)               # dLoss/dz at the output (sigmoid+BCE or softmax+CE)
            gW, gb = [None] * L, [None] * L
            for l in reversed(range(L)):
                gW[l], gb[l] = acts[l].T @ delta, delta.sum(0)
                if l > 0:
                    delta = (delta @ self.W[l].T) * (acts[l] > 0)   # back through ReLU
            for i, (p, g) in enumerate(zip(self.W + self.b, gW + gb)):   # Adam update (in place)
                m[i] = 0.9 * m[i] + 0.1 * g; v[i] = 0.999 * v[i] + 0.001 * g * g
                mh, vh = m[i] / (1 - 0.9 ** (ep + 1)), v[i] / (1 - 0.999 ** (ep + 1))
                p -= lr * mh / (np.sqrt(vh) + 1e-8)
        if len(self.sizes) > 2:
            self.hidden_scale = np.maximum(self.forward(X, return_all=True)[1].max(0), 1e-6)
        return history

for s in ([2, 1], [2, 3], [2, 8, 3]):
    print(f"Net({s}) -> {Net(s).n_params} trainable parameters")

## 2 · Drawing toolkit
These helpers are shared by every animation (no need to read them in class):

* `draw_network`: nodes and edges. **Edge width** = |weight|; solid grey = positive, dashed magenta = negative. When an input point is fed in, edges fade unless they carry a strong signal ($|a_i w_{ij}|$), hidden nodes glow **amber when ON** and turn **grey when OFF**, and output nodes fill with their class color in proportion to the predicted probability.
* `draw_regions`: the input plane shaded by predicted class probabilities, so a soft color blend means the model is uncertain.

In [ ]:
def layer_positions(sizes, dx=3.0):
    pos = []
    for l, n in enumerate(sizes):
        gap = min(1.6, max(sizes) / max(n, 1))
        ys = (np.arange(n)[::-1] - (n - 1) / 2) * (gap if n > 1 else 0)
        pos.append([(l * dx, y) for y in ys])
    return pos

def draw_network(ax, net, x=None, title=None, r=0.3):
    ax.clear(); ax.set_axis_off(); ax.set_aspect("equal")
    sizes, L = net.sizes, len(net.sizes)
    pos = layer_positions(sizes)
    acts = net.forward(np.asarray(x, float)[None, :], return_all=True) if x is not None else None
    mask = net.hidden_mask
    # ---- edges ----
    for l in range(L - 1):
        W = net.W[l]; wmax = np.abs(W).max() + 1e-9
        if acts is not None:
            signal = np.abs(acts[l][0][:, None] * W); smax = signal.max() + 1e-9
        for i in range(sizes[l]):
            for j in range(sizes[l + 1]):
                w = W[i, j]
                alpha = 0.55 if acts is None else 0.06 + 0.9 * signal[i, j] / smax
                if l == 1 and mask is not None and mask[i] == 0: alpha = 0.03
                (x0, y0), (x1, y1) = pos[l][i], pos[l + 1][j]
                ax.plot([x0, x1], [y0, y1], color=POS_W if w > 0 else NEG_W, ls="-" if w > 0 else (0, (3, 2)),
                        lw=0.4 + 3.0 * abs(w) / wmax, alpha=alpha, zorder=1, solid_capstyle="round")
    # ---- nodes ----
    for l, n in enumerate(sizes):
        for i, (px, py) in enumerate(pos[l]):
            text, tcol, ec, ls = "", "#222", "#555", "-"
            if l == 0:                                                    # input
                face = "#f4f4f4"; text = f"x{i+1}" if acts is None else f"{acts[0][0, i]:+.1f}"
            elif l == L - 1:                                              # output
                k = i if n > 1 else 1
                p = None if acts is None else acts[l][0, i]
                face = "white" if p is None else blend(CLASS_COLORS[k], p)
                ec = CLASS_COLORS[k]; text = "" if p is None else f"{p:.2f}"
            else:                                                         # hidden
                if mask is not None and mask[i] == 0:
                    face, ec, ls, text, tcol = "white", OFF_EDGE, ":", "✕", OFF_EDGE
                elif acts is None:
                    face, text = "#fff6e0", f"h{i+1}"
                else:
                    a = acts[l][0, i]
                    if a <= 0:
                        face, ec, text, tcol = OFF_FACE, OFF_EDGE, "off", "#888"
                    else:
                        t = a / (net.hidden_scale[i] if net.hidden_scale is not None else 1)
                        face, ec, text = blend(ON_COLOR, 0.35 + 0.65 * min(t, 1)), "#9a6a00", f"{a:.1f}"
                        ax.add_patch(Circle((px, py), r * 1.55, color=ON_COLOR, alpha=0.25, lw=0, zorder=2))
            ax.add_patch(Circle((px, py), r, facecolor=face, edgecolor=ec, lw=1.6, ls=ls, zorder=3))
            ax.text(px, py, text, ha="center", va="center", fontsize=7.5, color=tcol, zorder=4)
            if l == 0:
                ax.text(px - r - 0.15, py, f"x{i+1}", ha="right", va="center", fontsize=9)
            if l == L - 1:
                lab = "P(class B)" if n == 1 else f"class {CLASS_NAMES[i]}"
                ax.text(px + r + 0.15, py, lab, ha="left", va="center", fontsize=9)
    # ---- layer captions ----
    ymin = min(y for layer in pos for _, y in layer)
    out_name = "Output (sigmoid)" if sizes[-1] == 1 else "Output (softmax)"
    names = ["Input"] + ["Hidden (ReLU)"] * (L - 2) + [out_name]
    for l, nm in enumerate(names):
        ax.text(pos[l][0][0], ymin - 0.85, nm, ha="center", va="top", fontsize=9, color="#444")
    ax.set_xlim(-1.4, (L - 1) * 3.0 + 2.2); ax.set_ylim(ymin - 1.5, -ymin + 0.7)
    if title: ax.set_title(title, fontsize=11)

GRID_LIM = 3.0
_g = np.linspace(-GRID_LIM, GRID_LIM, 150)
GX, GY = np.meshgrid(_g, _g)
GRID = np.c_[GX.ravel(), GY.ravel()]

def draw_regions(ax, net, X=None, y=None, title=None, boundary=True):
    P = net.predict_proba(GRID); K = P.shape[1]
    cols = np.array([to_rgb(c) for c in CLASS_COLORS[:K]])
    rgb = 0.55 * (P @ cols) + 0.45                      # lightened probability-weighted class colors
    ax.imshow(rgb.reshape(GX.shape + (3,)), origin="lower", extent=[-GRID_LIM, GRID_LIM] * 2, zorder=0)
    if boundary:
        lab = P.argmax(1).reshape(GX.shape)
        ax.contour(GX, GY, lab, levels=np.arange(K) + 0.5, colors="#222", linewidths=1.2, zorder=1)
    if X is not None: plot_data(ax, X, y, alpha=0.85, s=16)
    ax.set_xlim(-GRID_LIM, GRID_LIM); ax.set_ylim(-GRID_LIM, GRID_LIM); ax.set_aspect("equal")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    if title: ax.set_title(title, fontsize=11)

# ---- Preview: the three architectures side by side ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 1.25]))
for ax, (s, t) in zip(axes, [([2, 1], "A · Logistic regression"), ([2, 3], "B · Two-layer network (no hidden)"),
                             ([2, 8, 3], "C · Three-layer network (hidden layer)")]):
    draw_network(ax, Net(s, seed=3), title=f"{t}\n{s}  ·  {Net(s).n_params} parameters")
plt.suptitle("The same building block, stacked differently (biases are not drawn)", y=1.02)
plt.tight_layout(); plt.show()

## 3 · Two reusable animations
1. **`animate_training`:** replays training. Left: decision regions; middle: weights (edge thickness grows as weights grow); right: loss and accuracy.
2. **`animate_probe`:** a **probe point** travels through the input plane and is fed through the trained network frame by frame. You watch nodes switch on and off as the point crosses each boundary.

In [ ]:
def animate_training(sizes, history, X, y, title, interval=120):
    net = Net(sizes)
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[1, 1.15, 1]))
    epochs = [h["epoch"] for h in history]; losses = [h["loss"] for h in history]; accs = [h["acc"] for h in history]
    def update(f):
        h = history[f]; net.load(h)
        a1.clear(); draw_regions(a1, net, X, y, title=f"Decision regions · epoch {h['epoch']}")
        draw_network(a2, net, title="Weights (width = |w|)")
        a3.clear()
        a3.plot(epochs[:f + 1], losses[:f + 1], color="#2a78d6", lw=2, label="loss")
        a3.plot(epochs[:f + 1], accs[:f + 1], color="#eb6834", lw=2, ls="--", label="accuracy")
        a3.set_xlim(0, epochs[-1]); a3.set_ylim(0, max(1.05, max(losses) * 1.05))
        a3.set_xlabel("epoch"); a3.legend(frameon=False, loc="center right")
        a3.set_title(f"loss {h['loss']:.3f}   ·   accuracy {h['acc']:.0%}", fontsize=11)
        fig.suptitle(title, fontsize=13)
    anim = animation.FuncAnimation(fig, update, frames=len(history), interval=interval)
    fig.subplots_adjust(left=0.04, right=0.98, top=0.80, bottom=0.12, wspace=0.22); show_anim(fig, anim)

def path_line(p0, p1, n):
    return np.linspace(p0, p1, n)
def path_arc(radius, t0, t1, n, center=(0, 0)):
    t = np.linspace(t0, t1, n); return np.c_[center[0] + radius * np.cos(t), center[1] + radius * np.sin(t)]

def animate_probe(net, X, y, path, title, right="bars", hidden_lines=False, interval=110):
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 5.2), gridspec_kw=dict(width_ratios=[1, 1.25, 0.8]))
    draw_regions(a1, net, X, y); bg_title = a1.get_title()
    def update(f):
        p = path[f]
        a1.clear(); draw_regions(a1, net, X, y)
        if hidden_lines:                       # each hidden node's on/off line: w1*x1 + w2*x2 + b = 0
            W, b = net.W[0], net.b[0]
            h = np.maximum(0, p @ W + b)
            if net.hidden_mask is not None: h = h * net.hidden_mask
            for j in range(W.shape[1]):
                w1, w2 = W[:, j]
                if abs(w2) > abs(w1):
                    xs = np.array([-GRID_LIM, GRID_LIM]); ys = -(w1 * xs + b[j]) / w2
                else:
                    ys = np.array([-GRID_LIM, GRID_LIM]); xs = -(w2 * ys + b[j]) / w1
                on = h[j] > 0
                a1.plot(xs, ys, color=ON_COLOR if on else "#777", lw=2.2 if on else 0.9,
                        ls="-" if on else ":", zorder=2)
            a1.set_xlim(-GRID_LIM, GRID_LIM); a1.set_ylim(-GRID_LIM, GRID_LIM)
        a1.plot(*path[max(0, f - 25):f + 1].T, color="#222", lw=1.2, alpha=0.5, zorder=4)
        a1.scatter(*p, s=170, marker="*", color="#111", edgecolor="white", lw=1.2, zorder=5)
        a1.set_title("Probe point ★ moving through input space" +
                     ("\n(amber lines = hidden nodes currently ON)" if hidden_lines else ""), fontsize=10)
        draw_network(a2, net, x=p, title=f"Forward pass for x = ({p[0]:+.1f}, {p[1]:+.1f})")
        a3.clear()
        if right == "sigmoid":                 # logistic regression: show where z lands on the sigmoid
            z = float(p @ net.W[0][:, 0] + net.b[0][0]); zz = np.linspace(-8, 8, 200)
            a3.plot(zz, sigmoid(zz), color="#444", lw=2); a3.axhline(0.5, color="#999", ls=":", lw=1)
            a3.axvline(0, color="#999", ls=":", lw=1)
            zc = np.clip(z, -8, 8); pz = sigmoid(z)
            a3.scatter([zc], [pz], s=140, color=blend(CLASS_COLORS[1], pz), edgecolor="#222", zorder=5)
            a3.set_xlabel("z = w1·x1 + w2·x2 + b"); a3.set_ylabel("σ(z)  =  P(class B)")
            state = "ON → class B" if pz > 0.5 else "OFF → class A"
            a3.set_title(f"z = {z:+.2f},  σ(z) = {pz:.2f}\nneuron is {state}", fontsize=10)
            a3.set_ylim(-0.05, 1.05)
        else:                                  # multi-class: bar chart of softmax output
            P = net.predict_proba(p[None, :])[0]; K = len(P)
            a3.barh(range(K)[::-1], P, color=[CLASS_COLORS[k] for k in range(K)], height=0.6)
            for k in range(K):
                a3.text(min(P[k] + 0.03, 0.85), K - 1 - k, f"{P[k]:.2f}", va="center", fontsize=9)
            a3.set_yticks(range(K)[::-1]); a3.set_yticklabels([f"class {CLASS_NAMES[k]}" for k in range(K)])
            a3.set_xlim(0, 1); a3.set_xlabel("predicted probability")
            a3.set_title(f"Prediction: class {CLASS_NAMES[P.argmax()]}", fontsize=10)
        fig.suptitle(title, fontsize=13)
    anim = animation.FuncAnimation(fig, update, frames=len(path), interval=interval)
    fig.subplots_adjust(left=0.05, right=0.98, top=0.78, bottom=0.12, wspace=0.25); show_anim(fig, anim)

---
# Part A · Logistic regression = a single neuron

$$\hat p = P(y = B \mid \mathbf{x}) = \sigma(z), \qquad z = w_1x_1 + w_2x_2 + b, \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

* The neuron computes a **weighted sum** $z$ and squashes it with the **sigmoid**.
* The decision boundary is where $\hat p = 0.5 \iff z = 0$, which is the **straight line** $w_1x_1 + w_2x_2 + b = 0$.
* On one side of the line the neuron is "**ON**" (predicts B), on the other side it is "**OFF**" (predicts A).

### A1 · Training animation
We start from deliberately bad weights so you can watch the line rotate and slide into place.

In [ ]:
lr_model = Net([2, 1], seed=0)
lr_model.W[0][:] = [[0.6], [-0.9]]; lr_model.b[0][:] = [1.0]       # a deliberately poor starting line
hist_lr = lr_model.fit(X2, y2, epochs=200, lr=0.05, n_frames=45)
print(f"Learned: w1 = {lr_model.W[0][0,0]:.2f}, w2 = {lr_model.W[0][1,0]:.2f}, b = {lr_model.b[0][0]:.2f}"
      f"   |  accuracy = {hist_lr[-1]['acc']:.0%}")
animate_training([2, 1], hist_lr, X2, y2, "Part A · Logistic regression learning a linear boundary")

### A2 · The neuron switching ON / OFF
The probe ★ walks from class A to class B and back. Watch three linked views:
the point crossing the line (left), the single output node filling with color (middle), and the point sliding along the sigmoid (right). **The neuron flips at z = 0.**

In [ ]:
path_A = np.vstack([path_line([-2.2, -1.8], [2.2, 1.9], 35),          # walk from class A into class B
                    path_arc(2.9, 0.71, 0.71 + 2 * np.pi, 50) * 0.75])   # then loop around, crossing the line twice
animate_probe(lr_model, X2, y2, path_A, "Part A · One sigmoid neuron switching on and off", right="sigmoid")

### A3 · Where logistic regression breaks
On the **core & quadrants** data, merge the four arcs into one class ("ring" vs "core"). No single straight line can separate a core from a ring around it, so logistic regression is stuck near the majority-class accuracy.

In [ ]:
yc_bin = (yc > 0).astype(int)                      # 0 = core, 1 = ring
lr_ring = Net([2, 1], seed=0); lr_ring.fit(Xc, yc_bin, epochs=300, lr=0.05)
acc = (lr_ring.predict(Xc) == yc_bin).mean()
fig, ax = plt.subplots(figsize=(5, 5))
draw_regions(ax, lr_ring, Xc, yc_bin, title=f"Logistic regression on core vs ring\naccuracy {acc:.0%} (majority class = {yc_bin.mean():.0%})")
plt.show()

---
# Part B · Two-layer network: many logistic neurons side by side

Put **K output neurons** next to each other, one per class, each with its own weights. Each computes $z_k = \mathbf{w}_k^\top\mathbf{x} + b_k$, and **softmax** turns the K scores into probabilities:

$$P(y=k\mid\mathbf{x}) = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

* This is **multinomial (softmax) logistic regression**. With K = 2 it reduces exactly to Part A, because only the difference $z_1 - z_0$ matters.
* There is still **no hidden layer**, so every boundary between two classes is the straight line $z_j = z_k$.

### B1 · Training animation on three blobs

In [ ]:
softmax_model = Net([2, 3], seed=4)
hist_sm = softmax_model.fit(X3, y3, epochs=250, lr=0.05, n_frames=45)
print(f"Parameters: {softmax_model.n_params}   |  accuracy = {hist_sm[-1]['acc']:.0%}")
animate_training([2, 3], hist_sm, X3, y3, "Part B · Input → output (softmax) network learning three linear boundaries")

### B2 · Output nodes taking turns switching on
The probe circles through all three classes. As it enters each region, **that class's output node lights up** while the others dim. Softmax makes the nodes compete: their probabilities always sum to 1.

In [ ]:
path_B = np.vstack([path_arc(1.7, -np.pi, np.pi, 60, center=(0.05, -0.1)), path_line([-1.65, -0.1], [0.05, -0.1], 12)])
animate_probe(softmax_model, X3, y3, path_B, "Part B · Softmax output nodes switching on and off", right="bars")

### B3 · Still linear: the same network on core & quadrants
Straight boundaries cannot carve out a central core, and they cannot handle arcs that alternate B, C, B, C. Because the layout is symmetric, any line that helps one arc hurts its mirror image, so the linear model does **no better than guessing** (about 33%).

In [ ]:
softmax_cc = Net([2, 3], seed=4); softmax_cc.fit(Xc, yc, epochs=400, lr=0.05)
acc_b = (softmax_cc.predict(Xc) == yc).mean()
fig, ax = plt.subplots(figsize=(5, 5))
draw_regions(ax, softmax_cc, Xc, yc, title=f"Two-layer network on core & quadrants\naccuracy {acc_b:.0%} (straight boundaries only)")
plt.show()

---
# Part C · Three-layer network: add a hidden layer

$$\mathbf{h} = \text{ReLU}(W^{(1)\top}\mathbf{x} + \mathbf{b}^{(1)}), \qquad P(y\mid\mathbf{x}) = \text{softmax}(W^{(2)\top}\mathbf{h} + \mathbf{b}^{(2)})$$

* Each hidden node $h_j$ is like a logistic-regression unit: it has its **own straight line** $w_{1j}x_1 + w_{2j}x_2 + b_j = 0$. It is **OFF** (0) on one side and **ON** (grows linearly) on the other.
* The output layer is exactly the **Part B model**, but it now reads the **hidden features** $\mathbf{h}$ instead of the raw $\mathbf{x}$.
* Combining many half-planes produces **bent, piecewise-linear boundaries**.

### C1 · Training animation (8 hidden ReLU nodes)

In [ ]:
mlp = Net([2, 8, 3], seed=1)
hist_mlp = mlp.fit(Xc, yc, epochs=1200, lr=0.03, n_frames=60)
print(f"Parameters: {mlp.n_params}   |  accuracy = {hist_mlp[-1]['acc']:.0%}")
animate_training([2, 8, 3], hist_mlp, Xc, yc, "Part C · Adding a hidden layer bends the boundaries", interval=100)

### C2 · Hidden nodes switching ON / OFF
The probe travels diagonally through the core, then once around the ring (B → C → B → C).
* **Left:** each hidden node's line. It is **amber** when that node is ON for the current point and dotted grey when OFF.
* **Middle:** hidden nodes glow when ON (brighter = larger activation) and grey out when OFF. Only edges carrying a strong signal stay visible.

Notice that the *pattern* of ON/OFF hidden nodes is different in each region. That pattern is the new representation the output layer classifies.

In [ ]:
d = 1.85 / np.sqrt(2)
path_C = np.vstack([path_line([-d, d], [d, -d], 30),                       # quadrant II → core → quadrant IV
                    path_arc(1.85, -np.pi / 4, -np.pi / 4 + 2 * np.pi, 64)])  # then once around the ring
animate_probe(mlp, Xc, yc, path_C, "Part C · Hidden ReLU nodes switching on and off", right="bars", hidden_lines=True, interval=100)

### C3 · What does each hidden node "see"?
Each panel shows one hidden node's activation over the input plane. **Grey = OFF** (output 0); amber = ON, darker means a larger value. Each node is a half-plane detector, essentially a logistic-regression unit with a ReLU in place of the sigmoid.

In [ ]:
H = mlp.sizes[1]
Hgrid = np.maximum(0, GRID @ mlp.W[0] + mlp.b[0])
from matplotlib.colors import LinearSegmentedColormap
amber = LinearSegmentedColormap.from_list("amber", ["#fff3d6", "#ffd98a", ON_COLOR, "#7a5200"])
amber.set_bad(OFF_FACE)
fig, axes = plt.subplots(2, (H + 1) // 2, figsize=(3.2 * ((H + 1) // 2), 6.6))
for j, ax in enumerate(axes.ravel()):
    if j >= H: ax.axis("off"); continue
    act = np.ma.masked_less_equal(Hgrid[:, j].reshape(GX.shape), 0)     # OFF region -> grey
    ax.imshow(act, origin="lower", extent=[-GRID_LIM, GRID_LIM] * 2, cmap=amber, vmin=0)
    ax.contour(GX, GY, Hgrid[:, j].reshape(GX.shape), levels=[1e-6], colors="#222", linewidths=1)
    plot_data(ax, Xc, yc, alpha=0.35, s=6)
    frac_on = (np.maximum(0, Xc @ mlp.W[0][:, j] + mlp.b[0][j]) > 0).mean()
    w_out = mlp.W[1][j]
    favors = CLASS_NAMES[int(np.argmax(w_out))]
    ax.set_title(f"h{j+1}: ON for {frac_on:.0%} of data\npushes toward class {favors}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
plt.suptitle(f"Hidden layer = {H} half-plane feature detectors (grey = OFF)", y=1.0)
plt.tight_layout(); plt.show()

### C4 · Knock-out experiment: switch one hidden node OFF at a time
We force one hidden node to 0 for every input and watch what happens to the decision regions. Some nodes are essential (accuracy collapses), while others are partly redundant.

In [ ]:
def animate_knockout(net, X, y, interval=1400):
    H = net.sizes[1]
    masks = [np.ones(H)] + [np.where(np.arange(H) == j, 0.0, 1.0) for j in range(H)] + [np.ones(H)]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1, 1.2]))
    def update(f):
        net.hidden_mask = masks[f]
        acc = (net.predict(X) == y).mean()
        off = np.where(masks[f] == 0)[0]
        lab = "all hidden nodes ON" if len(off) == 0 else f"h{off[0]+1} switched OFF"
        a1.clear(); draw_regions(a1, net, X, y, title=f"{lab}  ·  accuracy {acc:.0%}")
        draw_network(a2, net, title="✕ = node forced OFF")
        net.hidden_mask = None
    anim = animation.FuncAnimation(fig, update, frames=len(masks), interval=interval)
    fig.subplots_adjust(left=0.06, right=0.98, top=0.90, bottom=0.11, wspace=0.1); show_anim(fig, anim)

animate_knockout(mlp, Xc, yc)

### C5 · (Optional, interactive) Toggle hidden nodes yourself
Click the buttons to switch hidden nodes **on/off** and watch the decision regions update. Try to find the **smallest set of nodes** that still keeps accuracy above 90%. *(Uses `ipywidgets`, which is built into Colab.)*

In [ ]:
import ipywidgets as widgets
toggles = [widgets.ToggleButton(value=True, description=f"h{j+1}", button_style="warning",
                                layout=widgets.Layout(width="58px")) for j in range(mlp.sizes[1])]
out = widgets.Output()

def redraw(_=None):
    mask = np.array([t.value for t in toggles], float)
    for t in toggles: t.button_style = "warning" if t.value else ""
    mlp.hidden_mask = mask
    with out:
        out.clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1, 1.2]))
        acc = (mlp.predict(Xc) == yc).mean()
        draw_regions(a1, mlp, Xc, yc, title=f"{int(mask.sum())} of {len(mask)} hidden nodes ON  ·  accuracy {acc:.0%}")
        draw_network(a2, mlp, title="Click the buttons above to switch nodes")
        plt.tight_layout(); plt.show()
    mlp.hidden_mask = None

for t in toggles: t.observe(redraw, "value")
display(widgets.HBox(toggles), out)
redraw()

---
# Part D · The transition, side by side
Same data (core & quadrants) and the same code; only the number of hidden nodes changes. **0 hidden nodes = Part B (softmax logistic regression).**

In [ ]:
configs = [[2, 3], [2, 2, 3], [2, 4, 3], [2, 8, 3]]
fig, axes = plt.subplots(2, 4, figsize=(18, 9), gridspec_kw=dict(height_ratios=[0.8, 1]))
rows = []
for c, s in enumerate(configs):
    net = mlp if s == [2, 8, 3] else Net(s, seed=1)
    if net is not mlp: net.fit(Xc, yc, epochs=1200, lr=0.03)
    acc = (net.predict(Xc) == yc).mean()
    nh = 0 if len(s) == 2 else s[1]
    rows.append((str(s), nh, net.n_params, acc))
    draw_network(axes[0, c], net, title=f"{nh} hidden nodes  ·  {net.n_params} params")
    draw_regions(axes[1, c], net, Xc, yc, title=f"accuracy {acc:.0%}")
plt.suptitle("From logistic regression to a feed-forward network: adding hidden nodes", fontsize=14)
plt.tight_layout(); plt.show()

print(f"{'architecture':<14}{'hidden':>8}{'params':>9}{'accuracy':>10}")
for a, h, p, acc in rows: print(f"{a:<14}{h:>8}{p:>9}{acc:>10.0%}")

---
## Take-aways
1. **Logistic regression is a one-neuron network.** Weighted sum → sigmoid → a linear boundary where the neuron flips on/off.
2. **Stacking output neurons (Part B)** gives multi-class softmax regression, which still has only linear boundaries.
3. **A hidden layer (Part C)** inserts many logistic-regression-like units. Each switches on/off across its own line, and the output layer classifies the **on/off pattern**, which bends the overall boundary.
4. More hidden units → more pieces in the piecewise-linear boundary → more flexible (and more prone to over-fitting, which this notebook doesn't test because it uses no held-out data).

## Try it yourself
* **E1.** In Part C, change `Net([2, 8, 3])` to `Net([2, 3, 3])`. Part D shows that 4 hidden nodes already reach 100%. Look at the four lines in the C2 animation: which line separates which regions? Why do 2 hidden nodes fail?
* **E2.** Replace ReLU with a sigmoid in the hidden layer (edit `forward` and the back-prop line `(acts[l] > 0)` → `acts[l] * (1 - acts[l])`). Hidden nodes no longer switch fully off. What changes in the C2 animation?
* **E3.** Re-run C1 with different `seed=` values. Do you ever get a "dead" hidden node that is OFF for every training point? What does the knock-out animation show for that node?
* **E4.** Add a train/test split and compare `[2, 8, 3]` with `[2, 64, 3]`. Which over-fits?
* **E5.** Compare with scikit-learn: `sklearn.linear_model.LogisticRegression` and `sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(8,))`.